# Chapter 01-04 · pandas I: loading, selecting, filtering, and dtypes

**Label:** Optional  |  **Time:** ~50 minutes  |  **Difficulty:** gentle, with one lab that catches professionals

**Prerequisites:** 01-03. You should know what a shape is, what `axis=0` collapses, and the
difference between a view and a copy.

**Position in the learning path:** module 01, chapter 4 of 6. Before: **01-03** (NumPy). After:
**01-05** (grouping, joining, timestamps).

---

## Why this matters

pandas is where you will spend most of your time in this course, and where the most consequential
mistakes are made - not because the library is hard, but because **it will happily do the wrong
thing quietly**.

This chapter's failure lab is a file that loads without a single warning, whose largest
temperature is reported as `9,0` when the real maximum is `31,7`, and whose total rentals come out
as `12009801,0508701130940`. Nothing raises. Everything looks fine until somebody notices that a
number is absurd - and in a model, nobody notices.

The defence is one line of code and one habit, and this chapter installs both.

## What you will be able to do

By the end of this chapter you can:

1. **Load** a CSV and immediately establish whether every column has the type you expected.
2. **Distinguish** a Series from a DataFrame, and select by label and by position without guessing.
3. **Filter** rows with boolean masks, including several conditions and membership tests.
4. **Diagnose** a column silently loaded as text, and say which operations lie about it and which
   raise.
5. **Explain** why an edit made through a filtered subset can silently vanish.

## Warm-up: retrieve, do not reread

From memory:

1. What is the axis rule?
2. Does `arr[:5]` share memory with `arr`?
3. What is the two-line check that catches a wrong-axis standardisation?
4. In 01-02, what did `b = a` copy?

<br>

*Answers: (1) `axis=n` is the axis that disappears. (2) yes - a plain slice is a view; fancy and
boolean indexing give copies. (3) assert the column means are 0 and the column standard deviations
are 1. (4) the reference - one object with two names.*

## The situation

The bike stand's sensor data finally arrives as a CSV, exported by a supplier in Germany. It has
six rows, five columns, and no obvious problems.

We will load it, look at it, select from it, filter it - and then discover that two of its columns
are not what they appear to be.

**The question this chapter answers:** what do you check, in the first sixty seconds after
loading a file, so that nothing downstream is built on a misunderstanding?

In [ ]:
import io

import numpy as np
import pandas as pd

RAW_CSV = '''sensor,site,temp_c,rentals,reading_date
s1,north,"18,5",1200,2024-03-01
s2,north,"24,1",980,2024-03-01
s3,south,"31,7","1,050",2024-03-02
s4,south,"9,0",870,2024-03-02
s5,east,"27,3",1130,2024-03-03
s6,East ,"19,4",940,2024-03-03
'''

df = pd.read_csv(io.StringIO(RAW_CSV))     # StringIO lets us read text as if it were a file
df

Six rows, five columns, no errors, no warnings. It looks like a table of temperatures and rental
counts.

### DataFrame, Series, and the index

- A **DataFrame** is a table: several columns, each with a name, sharing one index.
- A **Series** is a single column: one array of values plus that index.
- The **index** is the row labels. By default it is `0, 1, 2, ...`, but it is not a position - it
  is a label that travels with the row through filters, sorts and joins.

Underneath, each column is essentially a NumPy array, which is why everything from 01-03 still
applies: `axis=0` still collapses rows, boolean masks still select, and dtypes still matter.
pandas adds the names, the index, and a policy for missing values.

**The index being labels rather than positions is the first thing that surprises people.** Filter
a DataFrame and the surviving rows keep their original labels - you saw `0, 2, 4` in the 01-01
solutions. That is a feature: it lets you trace a row back to where it came from. It also means
`df[3]` and "the fourth row" are different questions, which is why there are two accessors.

In [ ]:
print("shape :", df.shape)
print("columns:", list(df.columns))
print("index  :", list(df.index))
print()
df.info()

## The first sixty seconds after any load

`df.info()` is the single most valuable line in pandas. Read three things from it, in this order:

1. **The row count.** Is it the number of rows the file actually has? A silently truncated read,
   or a file with an unexpected header, shows up here.
2. **The non-null counts.** Any column with fewer non-nulls than rows has missing values, and you
   now know before it surprises you.
3. **The dtype of every column.** This is the one people skim, and it is the one that matters.

Look at the dtypes above. `temp_c` and `rentals` are **`str`** - text - not numbers. `reading_date`
is text too, not a date.

Nothing warned us. `read_csv` inferred a type per column, and a column containing `"18,5"` is not
a number in any language pandas speaks, so it stayed text. The German decimal comma did it.

We will come back to that. First, selecting and filtering - which all work perfectly on a column
of text, and that is precisely the problem.

In [ ]:
# Selecting columns: one bracket gives a Series, two brackets give a DataFrame.
print(type(df["site"]).__name__, "  <- df['site']")
print(type(df[["site"]]).__name__, "   <- df[['site']]")
print()
print(df[["sensor", "site"]].head(3))

The double-bracket distinction is not pedantry - it is the reason scikit-learn errors say
"expected 2D array, got 1D array". `X` must be a DataFrame (or 2-D array), so it is
`df[["temp_c"]]`; `y` is a Series, so it is `df["rentals"]`. You met exactly this in 00-01.

Read `df[["a", "b"]]` as "index this DataFrame with a **list** of column names", which is why the
result is a table.

In [ ]:
# .loc works with LABELS; .iloc works with POSITIONS.
print("df.loc[2, 'site']  ->", df.loc[2, "site"])
print("df.iloc[2, 1]      ->", df.iloc[2, 1])
print()
print("rows 1-3 by label (INCLUSIVE end):")
print(df.loc[1:3, ["sensor", "site"]])
print()
print("rows 1-3 by position (EXCLUSIVE end):")
print(df.iloc[1:3, [0, 1]])

**`.loc` slices are inclusive of the end label; `.iloc` slices are exclusive, like every other
Python slice.** That inconsistency is real and permanent, and it is worth saying out loud once:
`df.loc[1:3]` gives three rows, `df.iloc[1:3]` gives two.

The reason is that `.loc` works on labels, and with arbitrary labels - dates, names - "up to but
not including" is unusable, because you would have to know what the *next* label is. With
timestamps this becomes obvious: `df.loc["2024-03-01":"2024-03-02"]` means those two days, which
is what anyone would want.

Right now the index happens to be `0, 1, 2...` so labels and positions coincide. As soon as you
filter, sort or set a date index, they diverge - and code that used `.loc` when it meant `.iloc`
starts returning different rows without any error.

**The habit:** use `.loc` almost always, because you almost always mean "the row about sensor s3",
not "the row currently sitting third".

In [ ]:
# Filtering rows: a boolean mask, exactly as in NumPy.
mask = df["site"] == "north"
print("the mask:", mask.tolist())
print()
print(df.loc[mask, ["sensor", "site", "rentals"]])

In [ ]:
# Several conditions: & and |, and each condition needs its own brackets.
busy_north = df.loc[(df["site"] == "north") & (df["rentals"] != "980"), ["sensor", "rentals"]]
print(busy_north)
print()
print("isin   :", df.loc[df["site"].isin(["north", "south"]), "sensor"].tolist())
print("negate :", df.loc[~df["site"].isin(["north", "south"]), "sensor"].tolist())
print("text   :", df.loc[df["sensor"].str.startswith("s1"), "sensor"].tolist())

Three things that will save you an hour each:

- **`&` and `|`, not `and` and `or`.** The Python keywords work on single true/false values;
  pandas needs the elementwise operators. Using `and` raises "truth value of a Series is
  ambiguous", which is at least a clear error.
- **Every condition needs its own parentheses.** `a == 1 & b == 2` parses as `a == (1 & b) == 2`,
  because `&` binds tighter than `==`. This produces either an error or a wrong answer, depending
  on the types.
- **`~` negates a mask**, and `.isin([...])` tests membership - far better than chaining a
  sequence of `|`.

Notice the filter above compares `rentals` against the **string** `"980"`. That worked. It should
have been a number, and pandas did not object - because at this point `rentals` really is text.

---

## Failure lab: the file that loaded perfectly

Now ask the data three ordinary questions.

**Predict before running:** the temperatures are 18.5, 24.1, 31.7, 9.0, 27.3 and 19.4. What will
`df["temp_c"].max()` return? What will `df["rentals"].sum()` return? And what will
`df["temp_c"].mean()` do?

In [ ]:
print("hottest reading     :", repr(df["temp_c"].max()))
print("total rentals       :", repr(df["rentals"].sum()))
print("largest rental count:", repr(df["rentals"].max()))
try:
    print("mean temperature    :", df["temp_c"].mean())
except TypeError as exc:
    print("mean temperature    : TypeError -", exc)

### Diagnosis

**The hottest reading is reported as `'9,0'`.** The real maximum is 31.7.

**The total rentals come out as `'12009801,0508701130940'`** - every value stuck end to end.

**The largest rental count is `'980'`**, when 1200 is plainly larger.

**And the mean raises a `TypeError`.**

Three of these are wrong and one is an error, and that asymmetry is the lesson.

**Why `max` lies.** Comparing text is alphabetical, character by character. `'9'` comes after `'3'`
in the alphabet of characters, so `'9,0'` beats `'31,7'` - just as `'980'` beats `'1200'` because
`'9'` beats `'1'`. Every ranking, every sort, every "top ten" over this column is wrong, and none
of them complain.

**Why `sum` is absurd.** Adding strings concatenates them. The result is a single 22-character
string that no one would mistake for a total - *if anyone looked*. Assigned to a variable and
divided by a row count, it becomes an error three cells later, blamed on the wrong thing.

**Why `mean` was the friendly one.** There is no way to average text, so pandas refuses. The
operation that raised is the one that did you a favour. If you had only ever computed means, you
would have found this within seconds.

**The root cause is one character.** `"18,5"` uses the German decimal comma. pandas tried to make a
number of it, failed, and left the column as text - which is the correct and conservative
behaviour. The mistake was not pandas'; it was not looking at `.dtypes`.

### The fix, and a second trap inside it

In [ ]:
# Option 1 - repair the columns explicitly after loading.
clean = df.assign(
    temp_c=lambda d: pd.to_numeric(d["temp_c"].str.replace(",", ".", regex=False), errors="coerce"),
    rentals=lambda d: pd.to_numeric(d["rentals"].str.replace(",", "", regex=False), errors="coerce"),
    reading_date=lambda d: pd.to_datetime(d["reading_date"]),
    site=lambda d: d["site"].str.strip().str.lower(),
)

print(clean.dtypes.to_string())
print()
print("hottest      :", clean["temp_c"].max())
print("mean temp    :", round(clean["temp_c"].mean(), 2))
print("total rentals:", clean["rentals"].sum())
print("sites        :", clean["site"].unique().tolist())

Now `max` is 31.7, the mean exists, the total is a number, and `East ` with its capital and its
trailing space has joined `east`.

`errors="coerce"` turns anything unconvertible into `NaN` rather than raising. That is the right
default here **provided you then count them** - a column that silently becomes half missing is a
worse problem than the one you started with. `clean["temp_c"].isna().sum()` is the follow-up, and
02-04 is the chapter about it.

Now the trap. `read_csv` has a `decimal` parameter, so it looks as though one argument fixes
everything:

In [ ]:
tempting = pd.read_csv(io.StringIO(RAW_CSV), decimal=",")

print(tempting.dtypes.to_string())
print()
print("temp_c  :", tempting["temp_c"].tolist())
print("rentals :", tempting["rentals"].tolist(), "  <- look at the third value")

`decimal=","` fixed `temp_c` correctly - and destroyed `rentals`.

The row that read `"1,050"` was a **thousands separator**: one thousand and fifty. Told that comma
means decimal point, pandas read it as **1.05**. A count of 1,050 bike rentals became 1.05, in a
column that is now a perfectly respectable `float64` with no missing values and nothing to
suggest anything is wrong.

This is worse than the original bug. The original was visibly absurd; this one is plausible. A
model trained on it will treat one busy day as a near-zero day, and the only way to catch it is to
have looked at the values.

**The correct load** states both conventions explicitly - `decimal=","` and `thousands=","` cannot
both be a comma, which is exactly why this file is ambiguous and should be fixed at the source or
column by column, as we did above.

### Remedies

| Remedy | What it catches | Cost |
|---|---|---|
| `df.info()` immediately after every load | Every wrong dtype, and missing values | One line. Non-negotiable |
| `df.head()` and *read the values* | Separators, units, placeholder text like `-` or `N/A` | Ten seconds |
| `df.describe(include="all")` | Impossible minimums and maximums, single-valued columns | One line |
| Assert the dtypes you expect | This bug, permanently, in a script someone reruns next year | Two lines |
| Fix it at the export, not in the notebook | The whole class of problem | A conversation with whoever sent the file |

An assert is worth writing out, because it turns a habit into something that survives you:

```python
expected = {"temp_c": "float64", "rentals": "int64"}
for column, dtype in expected.items():
    assert str(clean[column].dtype) == dtype, f"{column} is {clean[column].dtype}, expected {dtype}"
```

---

## Failure lab 2: the edit that vanished

A second one, shorter, and the most-asked pandas question in the world.

You want to correct sensor s3's temperature. You filter to the south site, assign the new value,
and check the original.

**Predict before running:** does `data` end up changed?

In [ ]:
data = clean.copy()
south = data[data["site"] == "south"]      # a filtered subset
south["temp_c"] = 0.0                      # correct the reading... in the subset

print("the subset:", south["temp_c"].tolist())
print("the original:", data.loc[data["site"] == "south", "temp_c"].tolist(), "  <- unchanged")

### Diagnosis

The subset changed. The original did not. **The edit went nowhere**, silently.

`data[data["site"] == "south"]` returns a **new DataFrame** - boolean indexing always copies, as
you saw in NumPy in 01-03. Assigning into that copy modifies the copy, and the copy is discarded
when the cell ends.

Older pandas raised a `SettingWithCopyWarning` here, which everybody learned to ignore. Modern
pandas uses copy-on-write, which makes the behaviour *consistent* - a filtered result is always a
copy - and therefore makes this failure **completely silent**. Predictable is better than
warning-but-unpredictable, but it does mean nothing tells you your correction was lost.

**The fix: do the selection and the assignment in one `.loc` call**, so pandas knows you are
writing into the original.

In [ ]:
data = clean.copy()
data.loc[data["site"] == "south", "temp_c"] = 0.0

print("the original:", data.loc[data["site"] == "south", "temp_c"].tolist(), "  <- changed")

**The rule:** *if there are two square brackets on the left of the `=`, you are probably editing a
copy.* One `.loc[rows, columns] = value` is the form that works.

The same reasoning covers `.iloc`, and it is the same distinction as views versus copies in NumPy
and `b = a` in Python - the third appearance of one idea in three chapters, which is why it is
worth learning as an idea rather than three rules.

## Common misconceptions

**"If it loaded without an error, the data is fine."**
`read_csv` will parse almost anything. A column of text where you expected numbers, a date as a
string, a header row read as data, a numeric ID silently turned into a float - all load cleanly.
`.info()` is not optional.

**"`df[df.a > 1]` and `df.loc[df.a > 1]` are the same."**
For *reading*, yes. For *writing*, only the `.loc` form modifies the original. Use `.loc`
consistently and the distinction stops mattering.

**"Missing values are always `NaN`."**
They arrive as empty strings, `"N/A"`, `"-"`, `"null"`, `-999`, `0`, and whitespace. pandas
recognises a standard list and no more. A sentinel like `-999` is the dangerous kind - it is a
valid number, so it loads as one and quietly drags every average down. 02-04.

**"The index is just row numbers."**
It is labels. After a filter, the labels have gaps; after a sort, they are out of order; after
`set_index("date")`, they are dates. `.reset_index(drop=True)` renumbers when you genuinely want
positions.

**"`object` dtype means mixed types."**
It usually means *text*. In modern pandas a text column shows as `str`. Either way, if a column you
expect to be numeric is not numeric, something in it was unparseable - find out what before
coercing it away.

**"Chained indexing is fine as long as I only read."**
Reading is fine; the risk is that the habit carries over to writing. And a chained read on a large
frame builds an intermediate copy, which is wasteful. `.loc` in one call is both safer and faster.

---

## Exercises

Solutions: `solutions/01_python_bridge/01-04_pandas_basics_solutions.ipynb`.

### Quick understanding

**E1 (define).** What is the difference between a Series and a DataFrame, and what does
`df[["a"]]` give you that `df["a"]` does not?

**E2 (explain).** Name the three things you read from `df.info()`, in order, and what each one
would catch.

**E3 (explain).** Why did `mean()` raise while `max()` returned a wrong answer? Which behaviour
would you rather have, and why?

### Hand calculation

**E4 (calculate).** Sort these strings the way pandas would when the column is text:
`"1200", "980", "31,7", "9,0", "105"`. Then sort the numbers they represent. How many positions
differ?

**E5 (calculate).** A DataFrame has index `[0, 1, 2, 3, 4]`. After `d = df[df["x"] > 0]` the
surviving rows are the ones originally at 0, 3 and 4. Write down what `d.loc[3]` gives, what
`d.iloc[3]` gives, and what `d.loc[1]` gives.

### Coding

**E6 (code).** Write `load_and_check(csv_text, expected_dtypes)` that loads a CSV, repairs the
German decimal commas in the columns you expect to be floats, and asserts every expected dtype.
Test it on `RAW_CSV`, and test that it fails loudly on a column you deliberately mistype.

**E7 (code).** From `clean`, produce a DataFrame of the rows recorded on 2 or 3 March at a site
whose name starts with `s`, showing only `sensor`, `site` and `temp_c`, sorted by `temp_c`
descending.

### Interpretation

**E8 (interpret).** `clean["temp_c"].describe()` reports a minimum of 9.0 and a maximum of 31.7
over six readings. Give three questions you would ask before treating the 9.0 as a real
measurement.

### Debugging

**E9 (diagnose).** A colleague's script has run nightly for a year. Last night's file included one
row where the `rentals` field was `"1 050"` with a space. Nothing errored, and this morning the
dashboard shows total rentals for the month as a 200-character string. Explain the chain of
events, say which single line of defence would have stopped it, and where in the script it should
go.

### Exam and interview reasoning

**E10 (defend).** *"Why do you always call `.info()` after loading a file? Isn't that a waste of
time on a dataset you've used before?"* Answer in four sentences.

**E11 (design).** You will receive this same supplier's CSV every morning, automatically. Describe
three checks you would run on each file before it is allowed into the pipeline, and say what
should happen when one fails.

### Transfer to a different situation

**E12 (design).** A hospital exports patient records where `age` is sometimes `"unknown"`,
`weight_kg` uses a decimal comma, and `admitted` is `"1"`/`"0"` as text. For each column, say what
dtype you want, how you would get there, and one thing that could go wrong in the conversion that
would not raise an error.

### Explain it to someone non-technical

**E13 (explain).** In under 70 words, explain to the supplier why their CSV broke your analysis
and what you would like them to change. No jargon, and be specific about the fix.

### Optional challenge

**E14 (code + diagnose).** Write `profile(df)` that returns a one-row-per-column summary: dtype,
number of missing values, number of distinct values, and - for text columns - whether stripping
whitespace and lowercasing would reduce the number of distinct values. Run it on the *raw* `df`
and say which three problems it would have found in the first sixty seconds.

In [ ]:
# Your workspace. Still in memory: RAW_CSV, df, clean, data, pd, np, io.

## Mastery check

Without scrolling up, can you:

- [ ] Say what `df.info()` tells you and why it is the first thing you run? *(If not: "The first
      sixty seconds".)*
- [ ] Explain when `.loc` and `.iloc` diverge? *(If not: "Selecting".)*
- [ ] Say why `max()` on a text column is silently wrong? *(If not: "Failure lab".)*
- [ ] Explain why an edit through a filtered subset disappears? *(If not: "Failure lab 2".)*
- [ ] Write a boolean filter with two conditions, with the brackets right? *(If not: "Filtering".)*

## What should now feel instinctive

1. **`.info()` before anything else**, every single load, including files you have used before.
2. **Read the dtypes, not just the shape.** A number-shaped column is not a numeric column.
3. **`.loc[rows, columns]` in one call** - both for selecting and, especially, for assigning.
4. **`&`, `|`, `~`, and brackets round every condition.**
5. **An operation that raises did you a favour.** The ones that returned an answer are the ones to
   check.

## Flashcards

| Question | Answer |
|---|---|
| Series vs DataFrame | One column with an index vs a table of columns sharing an index |
| `df["a"]` vs `df[["a"]]` | Series vs one-column DataFrame - the reason scikit-learn wants double brackets for `X` |
| `.loc` vs `.iloc` | Labels vs positions. `.loc` slices include the end, `.iloc` excludes it |
| First line after any load | `df.info()` - row count, non-null counts, dtypes |
| Why is `max()` on text wrong? | It compares alphabetically, so `'9,0'` beats `'31,7'` |
| Why did `mean()` raise? | Text cannot be averaged - the operation that fails is the one that helps you |
| `errors="coerce"` | Unconvertible values become `NaN` - always count them afterwards |
| Why did the edit through a filter vanish? | Boolean indexing returns a copy; the assignment modified the copy |
| The rule for assignment | One `.loc[rows, cols] = value`. Two brackets on the left means trouble |
| `and` or `&`? | `&` and `|` for elementwise masks, with parentheses round each condition |

## Next

**01-05 · pandas II: grouping, joining, timestamps.**

You can now load a file and trust what is in it. The next chapter does the three things that turn
a table into an answer: `groupby` (which is how every error analysis in this course is done),
joins (and counting the rows before and after, because a join that multiplies rows is leakage
waiting to happen), and time - parsing it, indexing by it, and resampling it, which module 09 is
built on.

New terms are in [GLOSSARY.md](../../GLOSSARY.md).